# Facial Emotion Recognition — PyTorch

Multi-class image classification on the Kaggle dataset `samithsachidanandan/human-face-emotions`, comparing three models:

- **A.** Custom CNN from scratch (conv blocks + BatchNorm + Dropout)
- **B.** MobileNetV2 transfer learning (freeze → fine-tune)
- **C.** ResNet50 transfer learning (same recipe)

All heavy logic lives in the `src/` package so the notebook and the CLI (`python -m src.run_all`) share one code path. Set `SMOKE=False` below for the full run.

In [ ]:
import numpy as np, torch
from src import config as C, data as D, models as M, train as T, evaluate as E, inspect_data as I
from src.run_all import seed_everything, subset_per_class

# ---- run configuration ----
SMOKE  = False                 # True = fast subset/1-epoch check; False = full run
AUG    = False                # train-time augmentation on/off
MODELS = ['cnn', 'mobilenet_v2', 'resnet50']

seed_everything(C.SEED)
print('Device:', C.DEVICE)

## STEP 1 — Data inspection
Detects classes, counts images, checks channels/sizes, logs corrupt & duplicate files (quarantined), and saves figures to `figures/`.

In [ ]:
good, classes, corrupt, dup = I.run(do_quarantine=True)
len(good), classes

## STEP 2 — Preprocessing: stratified split + class weights
70/15/15 stratified split (seed=42). Class weights handle imbalance. Resize→224×224, grayscale→3ch, pixels→[0,1] (+ImageNet standardization for transfer models) happen inside the transforms.

In [ ]:
if SMOKE:
    good = subset_per_class(good, classes, 200)
    print(f'[SMOKE] using {len(good)} images')

train_recs, val_recs, test_recs = D.stratified_split(good, classes)
cw = D.class_weights(train_recs, classes)
print(f'train={len(train_recs)} val={len(val_recs)} test={len(test_recs)}')
print('class weights:', dict(zip(classes, [round(float(x),3) for x in cw])))

## STEP 3 & 4 — Train each model, then evaluate
Shared `train_model` uses Adam, CrossEntropyLoss (class-weighted), EarlyStopping, best-checkpointing and AMP. `evaluate_model` writes the confusion matrix, training curves, per-class report and a row in `results/results.csv`.

In [ ]:
epochs    = 1 if SMOKE else C.EPOCHS
ft_epochs = 1 if SMOKE else C.FINETUNE_EPOCHS
scenario  = 'aug' if AUG else 'baseline'
results = []

for name in MODELS:
    norm = M.NORMALIZE[name]
    train_ld, val_ld, test_ld = D.make_loaders(
        train_recs, val_recs, test_recs, classes, augment=AUG, normalize=norm)
    print(f'\n===== {name} (norm={norm}, aug={AUG}) =====')
    model, history, _ = T.train_model(
        name, train_ld, val_ld, len(classes), class_weights=cw,
        epochs=epochs, finetune_epochs=ft_epochs)
    metrics = E.evaluate_model(model, test_ld, classes, name,
                               scenario=scenario, history=history)
    results.append(metrics)

## Results summary

In [ ]:
import pandas as pd
pd.read_csv(C.RESULTS_CSV)